In [1]:
# Python version
import sys
print(sys.version)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:16:45) [MSC v.1942 64 bit (AMD64)]


In [2]:
# Import key libraries and check versions
import pandas as pd
import numpy as np
import sklearn
import irapy

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("irapy:", irapy.__version__)

pandas: 2.3.3
numpy: 2.2.5
sklearn: 1.8.0
irapy: 0.1.2


In [3]:
# Import functions
from sklearn.datasets import make_regression
from irapy import single_ira
from sklearn.linear_model import LinearRegression

In [4]:
# Create a linear dataset
X, y, coef = make_regression(
    n_samples=5000,
    n_features=8,
    noise = 50,
    bias = 100,
    n_informative=5,
    coef=True,
    random_state=42
)

In [5]:
# Check coefficients
coef_df = pd.DataFrame({
    "Predictor": [f"X{i}" for i in range(1,9,1)],
    "Coefficient": coef
})
coef_df

,Predictor,Coefficient
0,X1,0.000000
1,X2,83.172921
2,X3,34.543443
3,X4,46.401843
4,X5,0.000000
5,X6,5.060258
6,X7,0.000000
7,X8,70.536980


In [6]:
# Convert to dataframe
X_ = pd.DataFrame(X,columns = [f"X{i}" for i in range(1,9,1)])
y_ = pd.Series(y,name = 'y')

In [7]:
# Check the descriptive statistics
df = pd.concat([X_,y_],axis=1)
df_summary = df.describe().T.reset_index(drop=False)
df_summary

,index,count,mean,std,min,25%,50%,75%,max
0,X1,5000.0,-0.001666,1.003244,-3.655087,-0.683898,-0.016279,0.692589,3.602832
1,X2,5000.0,-0.014783,1.004720,-3.836656,-0.706716,0.007052,0.677056,3.727833
2,X3,5000.0,-0.010352,0.989298,-3.367806,-0.679370,-0.020188,0.658254,3.852731
3,X4,5000.0,0.027009,1.007826,-3.375579,-0.652517,0.034849,0.695301,3.611350
4,X5,5000.0,-0.008339,0.992714,-3.232565,-0.694453,-0.009230,0.638371,3.926238
5,X6,5000.0,0.012870,1.003341,-3.688365,-0.666626,0.017828,0.697636,4.479084
6,X7,5000.0,-0.009173,0.985457,-4.465604,-0.682760,-0.004933,0.669864,3.335268
7,X8,5000.0,-0.012838,1.013953,-3.922400,-0.701108,-0.012262,0.680556,3.377768
8,y,5000.0,99.426620,134.181787,-362.102084,8.604036,99.062326,191.880390,546.922325


In [8]:
df_summary.to_csv('linear_data_summary.csv',index=False)

In [9]:
# Build a linear regression model
model = LinearRegression().fit(X_, y_)
r2 = model.score(X_, y_)
print(f"Model trained — R² score on training data: {r2:.4f}")

Model trained — R² score on training data: 0.8599


In [10]:
# Run IRA Calculation

def ira_cal():
    column_name = ['N. background samp', 'N. interpl'] + list(X_.columns)
    result = []
    step = 1
    total_steps = 16

    for n in [50, 100, 200, 500]:
        for i in [50, 100, 200, 500]:
            print(f"Step {step}/{total_steps}: background_samples={n}, interpolation_points={i}")
            ira_result = single_ira(
                X_,
                model,
                num_interpol=i,
                num_background_samples=n,
                random_state=42
            )
            
            # check the result of the first one (n = 50; i = 50)
            if step==1:
                display(ira_result)
            
            result.append([n, i] + ira_result.iloc[:, 1].tolist())
            step += 1

    result_df = pd.DataFrame(result, columns=column_name)
    return result_df

ira_cal_result = ira_cal()

print("\n--- Final Single IRA Result Table ---")
display(ira_cal_result)

Step 1/16: background_samples=50, interpolation_points=50


,predictor,ira_value
0,X1,5.690476
1,X2,632.919482
2,X3,243.675154
3,X4,329.299348
4,X5,0.971003
5,X6,56.659573
6,X7,8.811392
7,X8,517.640752


Step 2/16: background_samples=50, interpolation_points=100
Step 3/16: background_samples=50, interpolation_points=200
Step 4/16: background_samples=50, interpolation_points=500
Step 5/16: background_samples=100, interpolation_points=50
Step 6/16: background_samples=100, interpolation_points=100
Step 7/16: background_samples=100, interpolation_points=200
Step 8/16: background_samples=100, interpolation_points=500
Step 9/16: background_samples=200, interpolation_points=50
Step 10/16: background_samples=200, interpolation_points=100
Step 11/16: background_samples=200, interpolation_points=200
Step 12/16: background_samples=200, interpolation_points=500
Step 13/16: background_samples=500, interpolation_points=50
Step 14/16: background_samples=500, interpolation_points=100
Step 15/16: background_samples=500, interpolation_points=200
Step 16/16: background_samples=500, interpolation_points=500

--- Final Single IRA Result Table ---


,N. background samp,N. interpl,X1,X2,X3,X4,X5,X6,X7,X8
0,50,50,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
1,50,100,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
2,50,200,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
3,50,500,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
4,100,50,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
5,100,100,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
6,100,200,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
7,100,500,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
8,200,50,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752
9,200,100,5.690476,632.919482,243.675154,329.299348,0.971003,56.659573,8.811392,517.640752


In [11]:
ira_cal_result.to_csv('linear_example_ira_result.csv',index=False)